# ConsultaAI — Etapa 1.2: Instanciação, Inferência e Benchmark de Modelos

Este notebook carrega o dataset fatiado na Etapa 1.1, instancia os modelos de ASR (Wav2Vec2, Meta MMS e Whisper), executa a inferência e calcula as métricas de acurácia (WER/CER) em relação aos Gabaritos A (coloquial) e B (normalizado), além de medir tempos de execução.

In [ ]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Adiciona a pasta src ao sys.path para importar os módulos locais
sys.path.append(str(Path("..").resolve()))

from src.models.inference import ASRModelWrapper
from src.evaluation.metrics import calculate_wer, calculate_cer

## 1. Carregamento dos Metadados Preprocessados

Carregamos o arquivo `metadata.csv` contendo a localização dos áudios fatiados e as referências textuais.

In [ ]:
metadata_path = Path("../data/output/metadata.csv")
df = pd.read_csv(metadata_path)
print(f"Total de segmentos disponíveis para benchmark: {len(df)}")
df.head()

## 2. Seleção de Amostra para Avaliação Rápida

Como transcrever as 9.5 horas de áudio (16.963 segmentos) pode demorar bastante em CPU, vamos selecionar um subset para o benchmark inicial. 

Por padrão, usaremos os primeiros 100 segmentos do arquivo `besqau01`, mas você pode alterar para o tamanho de amostra que desejar.

In [ ]:
# Filtra segmentos de besqau01 como amostra estável de teste (ou use df.sample(N))
df_sample = df[df["file_id"] == "besqau01"].head(100).copy()
print(f"Tamanho da amostra para o benchmark: {len(df_sample)} segmentos")

## 3. Avaliação do Modelo Whisper (base)

Instanciamos o Whisper Base e rodamos a inferência na nossa amostra.

In [ ]:
# Instancia o modelo Whisper via faster-whisper
whisper_model = ASRModelWrapper(model_type="whisper", model_id_or_size="base")

In [ ]:
whisper_preds = []
whisper_times = []
audio_durations = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Whisper Base"):
    audio_path = Path("..") / row["audio_path"]
    try:
        res = whisper_model.transcribe(audio_path)
        whisper_preds.append(res["text"])
        whisper_times.append(res["execution_time_s"])
        audio_durations.append(res["audio_len_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        whisper_preds.append("")
        whisper_times.append(0.0)
        audio_durations.append(0.0)

df_sample["pred_whisper_base"] = whisper_preds
df_sample["time_whisper_base"] = whisper_times
df_sample["audio_len"] = audio_durations

## 4. Avaliação do Modelo Wav2Vec2 (Português)

Instanciamos o Wav2Vec2 treinado para o português (`jonatasgrosman/wav2vec2-large-xlsr-53-portuguese`) via Hugging Face e rodamos a inferência.

In [ ]:
# Instancia o modelo Wav2Vec2
wav2vec2_model = ASRModelWrapper(
    model_type="wav2vec2", 
    model_id_or_size="jonatasgrosman/wav2vec2-large-xlsr-53-portuguese"
)

In [ ]:
wav2vec2_preds = []
wav2vec2_times = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Wav2Vec2 PT"):
    audio_path = Path("..") / row["audio_path"]
    try:
        res = wav2vec2_model.transcribe(audio_path)
        wav2vec2_preds.append(res["text"])
        wav2vec2_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        wav2vec2_preds.append("")
        wav2vec2_times.append(0.0)

df_sample["pred_wav2vec2_pt"] = wav2vec2_preds
df_sample["time_wav2vec2_pt"] = wav2vec2_times

## 5. Cálculo de Métricas Comparativas (WER/CER)

Calculamos as taxas de erro das predições de cada modelo comparando contra as referências dos Gabaritos A (coloquial) e B (normalizado).

In [ ]:
# Filtra linhas válidas (onde a inferência funcionou)
df_valid = df_sample[df_sample["audio_len"] > 0].copy()

# Referências
ref_a = df_valid["text_gabarito_a"].fillna("").tolist()
ref_b = df_valid["text_gabarito_b"].fillna("").tolist()

# Predições
pred_whisper = df_valid["pred_whisper_base"].fillna("").tolist()
pred_wav2vec2 = df_valid["pred_wav2vec2_pt"].fillna("").tolist()

# WER
wer_whisper_a = calculate_wer(ref_a, pred_whisper)
wer_whisper_b = calculate_wer(ref_b, pred_whisper)
wer_wav2vec2_a = calculate_wer(ref_a, pred_wav2vec2)
wer_wav2vec2_b = calculate_wer(ref_b, pred_wav2vec2)

# CER
cer_whisper_a = calculate_cer(ref_a, pred_whisper)
cer_whisper_b = calculate_cer(ref_b, pred_whisper)
cer_wav2vec2_a = calculate_cer(ref_a, pred_wav2vec2)
cer_wav2vec2_b = calculate_cer(ref_b, pred_wav2vec2)

# Métricas de Hardware
total_audio_s = df_valid["audio_len"].sum()
rtf_whisper = df_valid["time_whisper_base"].sum() / total_audio_s
rtf_wav2vec2 = df_valid["time_wav2vec2_pt"].sum() / total_audio_s

# Montando o Report
results = {
    "Modelo": ["Whisper Base", "Wav2Vec2 PT (XLSR-53)"],
    "WER - Gabarito A (Coloquial)": [wer_whisper_a, wer_wav2vec2_a],
    "WER - Gabarito B (Normalizado)": [wer_whisper_b, wer_wav2vec2_b],
    "CER - Gabarito A (Coloquial)": [cer_whisper_a, cer_wav2vec2_a],
    "CER - Gabarito B (Normalizado)": [cer_whisper_b, cer_wav2vec2_b],
    "RTF (Fator de Tempo Real)": [rtf_whisper, rtf_wav2vec2],
    "Tempo Médio por Segmento (s)": [df_valid["time_whisper_base"].mean(), df_valid["time_wav2vec2_pt"].mean()]
}

df_results = pd.DataFrame(results)
df_results

## 6. Análise de Amostras Lado a Lado

Vamos visualizar algumas predições reais comparadas com as referências originais para entender as características e tipos de erros de cada modelo.

In [ ]:
columns_to_show = [
    "speaker",
    "text_gabarito_a",
    "text_gabarito_b",
    "pred_whisper_base",
    "pred_wav2vec2_pt"
]
df_valid[columns_to_show].head(10)

## 7. Salvamento do Dataset de Saída Comparativo

Salvamos as predições completas da amostra em formato CSV para visualização dos resultados


In [ ]:
output_csv_path = Path("../data/output/benchmark_predictions.csv")
df_valid.to_csv(output_csv_path, index=False, encoding="utf-8")
print(f"Predições detalhadas salvas com sucesso em:\n{output_csv_path.resolve()}")